# Next Word Prediction Using LSTM and GRU
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1. **Data Collection**: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.
2. **Data Preprocessing**: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.
3. **Model Building**: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.
4. **Model Training**: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.
5. **Model Evaluation**: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.
6. **Deployment**: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

In [5]:
import nltk
nltk.download('gutenberg')

/home/htet-aung-lynn/Study/E2E-Predict-Next-Word-with-LSTM-GRU/venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/htet-aung-lynn/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package gutenberg to /home/htet-aung-
[nltk_data]     lynn/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


True

## Import libraries

In [1]:
# Essential libraries
import pandas as pd
import numpy as np

# NLP libraries
from nltk.corpus import gutenberg

# TensorFlow libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM, GRU, Dropout


# Scikit-learn libraries
from sklearn.model_selection import train_test_split as TTS


## Load dataset and saved in local

In [6]:
# Load data
data = gutenberg.raw('shakespeare-hamlet.txt')

# Save data into local
with open("data/hamlet.txt", "w", encoding='utf-8') as file:
    file.write(data)

## Data preprocessing

### Load dataet

In [2]:
# Load dataset
with open('data/hamlet.txt', 'r') as file:
    text = file.read()

print(text)

FileNotFoundError: [Errno 2] No such file or directory: 'data/hamlet.txt'

### Tokenize text

```python
text = "the food is good the food is delicious"
```

The tokenizer builds a vocabulary similar to this:

Word        | Frequency | Index |
------------| ----------|-------|
the         |    2      |   1   |
food        |    2      |   2   |
is          |    1      |   3   |
good        |    1      |   4   |
delocious   |    1      |   5   |


In [ ]:
tokenizer = Tokenizer()                    # converts text to lowercase and removes certain punctuation
tokenizer.fit_on_texts([text])             # analyzes the input text and builds a vocabulary
TOTAL_WORDS = len(tokenizer.word_index)+1 
TOTAL_WORDS

4818

In [ ]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'of': 4,
 'i': 5,
 'you': 6,
 'a': 7,
 'my': 8,
 'it': 9,
 'in': 10,
 'that': 11,
 'ham': 12,
 'is': 13,
 'not': 14,
 'his': 15,
 'this': 16,
 'with': 17,
 'your': 18,
 'but': 19,
 'for': 20,
 'me': 21,
 'lord': 22,
 'as': 23,
 'what': 24,
 'he': 25,
 'be': 26,
 'so': 27,
 'him': 28,
 'haue': 29,
 'king': 30,
 'will': 31,
 'no': 32,
 'our': 33,
 'we': 34,
 'on': 35,
 'are': 36,
 'if': 37,
 'all': 38,
 'then': 39,
 'shall': 40,
 'by': 41,
 'thou': 42,
 'come': 43,
 'or': 44,
 'hamlet': 45,
 'good': 46,
 'do': 47,
 'hor': 48,
 'her': 49,
 'let': 50,
 'now': 51,
 'thy': 52,
 'how': 53,
 'more': 54,
 'they': 55,
 'from': 56,
 'enter': 57,
 'at': 58,
 'was': 59,
 'oh': 60,
 'like': 61,
 'most': 62,
 'there': 63,
 'well': 64,
 'know': 65,
 'selfe': 66,
 'would': 67,
 'them': 68,
 'loue': 69,
 'may': 70,
 "'tis": 71,
 'vs': 72,
 'sir': 73,
 'qu': 74,
 'which': 75,
 'did': 76,
 'why': 77,
 'laer': 78,
 'giue': 79,
 'thee': 80,
 'ile': 81,
 'must': 82,
 'hath': 

### Create input sequences

In [ ]:
text.split('\n')

['[The Tragedie of Hamlet by William Shakespeare 1599]',
 '',
 '',
 'Actus Primus. Scoena Prima.',
 '',
 'Enter Barnardo and Francisco two Centinels.',
 '',
 "  Barnardo. Who's there?",
 '  Fran. Nay answer me: Stand & vnfold',
 'your selfe',
 '',
 '   Bar. Long liue the King',
 '',
 '   Fran. Barnardo?',
 '  Bar. He',
 '',
 '   Fran. You come most carefully vpon your houre',
 '',
 "   Bar. 'Tis now strook twelue, get thee to bed Francisco",
 '',
 "   Fran. For this releefe much thankes: 'Tis bitter cold,",
 'And I am sicke at heart',
 '',
 '   Barn. Haue you had quiet Guard?',
 '  Fran. Not a Mouse stirring',
 '',
 '   Barn. Well, goodnight. If you do meet Horatio and',
 'Marcellus, the Riuals of my Watch, bid them make hast.',
 'Enter Horatio and Marcellus.',
 '',
 "  Fran. I thinke I heare them. Stand: who's there?",
 '  Hor. Friends to this ground',
 '',
 '   Mar. And Leige-men to the Dane',
 '',
 '   Fran. Giue you good night',
 '',
 "   Mar. O farwel honest Soldier, who hath reli

In [ ]:
txt = "Ham. Oh Villany! How? Let the doore be lock'd."

# Converts text into numerical sequences based on the tokenizer's word_index dictionary
tk_list = tokenizer.texts_to_sequences([txt])[0]
print("Token list:", tk_list)

for i in range(1,len(tk_list)):
        print(tk_list[:i+1])

Token list: [12, 60, 4728, 53, 50, 1, 868, 26, 4729]
[12, 60]
[12, 60, 4728]
[12, 60, 4728, 53]
[12, 60, 4728, 53, 50]
[12, 60, 4728, 53, 50, 1]
[12, 60, 4728, 53, 50, 1, 868]
[12, 60, 4728, 53, 50, 1, 868, 26]
[12, 60, 4728, 53, 50, 1, 868, 26, 4729]


In [ ]:
input_seq = []
for line in text.split('\n'):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for _ in range(1, len(tokens)):
        input_seq.append(tokens[:_+1])

input_seq

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

In [ ]:
max([len(x) for x in input_seq])

14

In [ ]:
# Get Max words for pad sequences
MAX_SEQ = max(map(len, input_seq)) # same with "max([len(x) for x in input_seq])"
MAX_SEQ

14

In [ ]:
input_seq = pad_sequences(sequences=input_seq, maxlen=MAX_SEQ, padding='pre')
input_seq

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]],
      shape=(25732, 14), dtype=int32)

### Prepare data for splitting

In [ ]:
# Sparate independent and dependent data
X = input_seq[:,:-1] # take all words, except last word, as independent feature
y = input_seq[:,-1]  # take last word as dependent feature

In [ ]:
X.shape, y.shape

((25732, 13), (25732,))

In [ ]:
y = to_categorical(x=y, num_classes=TOTAL_WORDS) # TOTAL_WORDS = 4818
y.shape

(25732, 4818)

In [ ]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(25732, 4818))

In [ ]:
# Train-Test-Split data
X_train, X_test, y_train, y_test = TTS(X, y, test_size=0.2)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((20585, 13), (5147, 13), (20585, 4818), (5147, 4818))

In [ ]:
# Define early stopping
early_stopping = EarlyStopping(monitor='val_loss',         # quantity to be monitored
                               patience=3,                 # number of epochs with no improvement after which training will be stopped
                               restore_best_weights=True)  # whether to restore model weights from the epoch with the best value of the monitored quantity

In [3]:
# Define model
model = Sequential()
model.add(Embedding(input_dim=TOTAL_WORDS,    # size of the vocabulary, including the reserved index for padding
                    output_dim=100,           # number of dimensions in each word's embedding vector
                    input_length=MAX_SEQ-1))  # intended length of the input sequence
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(TOTAL_WORDS, activation='softmax'))

NameError: name 'TOTAL_WORDS' is not defined

In [19]:
## Train our LSTM RNN

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,GRU

## Define the model
model=Sequential()
model.add(Embedding(total_words,100,input_length=max_sequence_len-1))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation="softmax"))

# #Compile the model
model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 13, 100)           481800    
                                                                 
 lstm_3 (LSTM)               (None, 13, 150)           150600    
                                                                 
 dropout_1 (Dropout)         (None, 13, 150)           0         
                                                                 
 lstm_4 (LSTM)               (None, 100)               100400    
                                                                 
 dense_1 (Dense)             (None, 4818)              486618    
                                                                 
Total params: 1219418 (4.65 MB)
Trainable params: 1219418 (4.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
## GRU RNN
## Define the model
model=Sequential()
model.add(Embedding(total_words,100,input_length=max_sequence_len-1))
model.add(GRU(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(100))
model.add(Dense(total_words,activation="softmax"))

# #Compile the model
model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
model.summary()

In [20]:
## Train the model
history=model.fit(x_train,y_train,epochs=50,validation_data=(x_test,y_test),verbose=1,callbacks=[early_stopping])


Epoch 1/50


644/644 [==============================] - 20s 24ms/step - loss: 6.8965 - accuracy: 0.0334 - val_loss: 6.7613 - val_accuracy: 0.0326
Epoch 2/50
644/644 [==============================] - 15s 23ms/step - loss: 6.4683 - accuracy: 0.0386 - val_loss: 6.8442 - val_accuracy: 0.0350
Epoch 3/50
644/644 [==============================] - 15s 23ms/step - loss: 6.3255 - accuracy: 0.0463 - val_loss: 6.8853 - val_accuracy: 0.0418
Epoch 4/50
644/644 [==============================] - 15s 23ms/step - loss: 6.1834 - accuracy: 0.0541 - val_loss: 6.9457 - val_accuracy: 0.0455
Epoch 5/50
644/644 [==============================] - 14s 22ms/step - loss: 6.0507 - accuracy: 0.0552 - val_loss: 6.9807 - val_accuracy: 0.0459
Epoch 6/50
644/644 [==============================] - 15s 23ms/step - loss: 5.9159 - accuracy: 0.0636 - val_loss: 7.0333 - val_accuracy: 0.0509
Epoch 7/50
644/644 [==============================] - 15s 23ms/step - loss: 5.7781 - accuracy: 0.0734 - val_loss: 7.0298 - val_accurac

In [21]:
# Function to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]  # Ensure the sequence length matches max_sequence_len-1
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [22]:
input_text="To be or not to be"
print(f"Input text:{input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(f"Next Word PRediction:{next_word}")

Input text:To be or not to be
Next Word PRediction:considered


In [24]:
## Save the model
model.save("next_word_lstm.h5")
## Save the tokenizer
import pickle
with open('tokenizer.pickle','wb') as handle:
    pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)

In [27]:
input_text="  Barn. Last night of all,When yond same"
print(f"Input text:{input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(f"Next Word PRediction:{next_word}")

Input text:  Barn. Last night of all,When yond same
Next Word PRediction:cosin
